# Comment Reaction Analysis: AI vs Human Posts

This notebook classifies comments from AI-generated and human-created posts using a **local LLM via Ollama**.

**Pipeline:**
1. Load two CSVs and parse the wide/transposed format
2. Preview data and define reaction categories
3. Classify each comment locally via Ollama

## 0. Install dependencies

In [ ]:
%pip install pandas matplotlib seaborn scipy tqdm --quiet


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 1. Configuration

In [ ]:
import os, json, time, re
import pandas as pd
from tqdm.notebook import tqdm
import urllib.request
import warnings
warnings.filterwarnings('ignore')

# ── Paths ─────────────────────────────────────────────────────────────────────
AI_CSV_PATH    = "data/ai_posts_comments.csv"
HUMAN_CSV_PATH = "data/human_posts_comments.csv"
ROW_ID_COL     = "Post ID (DO NOT EDIT COL)"

# ── Ollama settings ───────────────────────────────────────────────────────────
MODEL        = "llama3.2"
OLLAMA_URL   = "http://localhost:11434/api/generate"
SLEEP_BETWEEN_CALLS = 0.0

print(f"Model : {MODEL}")
print(f"URL   : {OLLAMA_URL}")
print("No API key needed — Ollama runs locally.")

Model : llama3.2
URL   : http://localhost:11434/api/generate
No API key needed — Ollama runs locally.


## 2. Load & reshape data

The CSVs are in **wide / transposed** format:
- **First column** = row identifier
- **Remaining columns** = one column per post
- **Cell values** = comment text (empty = no comment for that post/slot)

We melt this into long form: one row per comment.

In [3]:
def load_wide_csv(path, condition_label, row_id_col=ROW_ID_COL):
    """
    Read a wide-format CSV and return a long-form DataFrame:
      condition | post_id | comment_slot | comment
    """
    raw = pd.read_csv(path, dtype=str)
    print(f"[{condition_label}] Raw shape: {raw.shape}  "
          f"({raw.shape[1]-1} post columns, {raw.shape[0]} comment-slot rows)")

    # Identify row-id column
    if row_id_col in raw.columns:
        id_col    = row_id_col
        post_cols = [c for c in raw.columns if c != row_id_col]
    else:
        id_col    = raw.columns[0]
        post_cols = raw.columns[1:].tolist()
        print(f"  WARNING: ROW_ID_COL not found; using first column '{id_col}' instead.")

    # Drop all-empty trailing columns (Excel/Sheets padding artefacts)
    raw[post_cols] = raw[post_cols].replace(r'^\s*$', float('nan'), regex=True)
    post_cols = [c for c in post_cols if raw[c].notna().any()]

    # Melt wide -> long
    long = (
        raw[[id_col] + post_cols]
        .melt(id_vars=id_col, value_vars=post_cols,
              var_name="post_id", value_name="comment")
        .rename(columns={id_col: "comment_slot"})
    )

    # Remove empty cells
    long = long.dropna(subset=["comment"])
    long["comment"] = long["comment"].str.strip()
    long = long[long["comment"] != ""].copy()

    long.insert(0, "condition", condition_label)
    long = long.reset_index(drop=True)
    print(f"  -> {len(long)} non-empty comments across {long['post_id'].nunique()} posts.")
    return long


ai_df    = load_wide_csv(AI_CSV_PATH,    "AI")
human_df = load_wide_csv(HUMAN_CSV_PATH, "Human")

all_df = pd.concat([ai_df, human_df], ignore_index=True)
print(f"\nTotal comments: {len(all_df)}  (AI: {len(ai_df)}, Human: {len(human_df)})")
all_df.head(10)

[AI] Raw shape: (40, 175)  (174 post columns, 40 comment-slot rows)
  -> 1489 non-empty comments across 172 posts.
[Human] Raw shape: (40, 184)  (183 post columns, 40 comment-slot rows)
  -> 1828 non-empty comments across 183 posts.

Total comments: 3317  (AI: 1489, Human: 1828)


,condition,comment_slot,post_id,comment
0,AI,C-44eycs1H5,1,So beautiful ❣️
1,AI,DVMb9kqElDz,1,Simply outstanding 🔥🔥🔥
2,AI,DNSWhv1uGjZ,1,"Oh darling, say👏it👏louder👏. Your vibe and art ..."
3,AI,DMyBan2tx-f,1,❤️❤️❤️
4,AI,DHOa6Ajt1dq,1,Adore this job 😍 sea horse so beautiful 💫
5,AI,C0sQkLEPkyC,1,That's why the kids have to be in bed! 😮👏👏
6,AI,DQcyILfj1Nx,1,🤍🤍🌟🙌🖤
7,AI,DQhcx5VDMHT,1,😮
8,AI,DVYkAyEDF9p,1,😍😍😍
9,AI,Co-OeX8tLU-,1,🔥


## 3. Preview a sample of comments

Inspect a few comments per condition before running the full classification 

In [4]:
n_sample = min(8, len(ai_df), len(human_df))
sample   = all_df.groupby("condition", group_keys=False).sample(n=n_sample, random_state=42)

for cond, grp in sample.groupby("condition"):
    print(f"\n{'='*60}\n  CONDITION: {cond}\n{'='*60}")
    for _, row in grp.iterrows():
        print(f"  [post {row['post_id']}]  {row['comment'][:180]}")
        print(f"  {'-'*56}")


  CONDITION: AI
  [post 31]  🔥🔥🔥🔥🔥🔥🔥
  --------------------------------------------------------
  [post 8]  Perhaps freedom can only be taken away...❤️❤️❤️dearest Farsan the sun shines for all... the moon shines for all let the wind lift us... I love your work always refreshing and uplif
  --------------------------------------------------------
  [post 7]  Beklerken kök saldım demek eylemini anlatan bi görsel gibi... Oldukça köklü ve sabırlı 🪾
  --------------------------------------------------------
  [post 31]  Nice
  --------------------------------------------------------
  [post 36]  Traumhaft schön ❤️❤️❤️❤️❤️
  --------------------------------------------------------
  [post 24]  May no one find me , not even in my silence 🕊️
  --------------------------------------------------------
  [post 12]  ❤️
  --------------------------------------------------------
  [post 23]  So beautiful! ☺️
  --------------------------------------------------------

  CONDITION: Human
  [post 14] 

## 4. Define reaction categories

In [ ]:
CATEGORIES = {
    "praise_admiration":      "Positive, complimentary, or admiring reaction expressed in words (e.g. 'beautiful', 'amazing work', 'I love this', 'stunning').",
    "emoji_only":             "Comment consists entirely or almost entirely of emoji/symbols with no meaningful text (e.g. '😍❤️🔥', '👏👏👏').",
    "authenticity_policing":  "Questions or denies the legitimacy of the work as real art (e.g. 'this isn't real art', 'AI can't be creative', 'just pressing buttons').",
    "ethics_copyright":       "Raises concerns about ethics, copyright, or harm to human artists (e.g. 'you stole from artists', 'AI training is theft').",
    "hostility_harassment":   "Aggressive, rude, dismissive, or harassing tone directed at the creator or the work.",
    "support_defense":        "Defends AI art or the creator, or pushes back on criticism (e.g. 'AI is just a tool', 'all art borrows from others').",
    "neutral_question":       "Neutral comment or factual question with no strong sentiment (e.g. 'What app did you use?', 'What is Niji?', asking for prompts).",
    "humor_meme":             "Joke, pun, meme reference, or clearly light-hearted/ironic comment.",
    "needs_review":           "Comment meaning is genuinely unclear — ambiguous slang, mixed/foreign language, or inside reference with no clear sentiment.",
    "non_english":            "Comment is clearly written in a language other than English.",
    "other":                  "Does not fit any category above.",
}

CATEGORY_KEYS = list(CATEGORIES.keys())
print(f"{len(CATEGORY_KEYS)} categories defined:")
for k, v in CATEGORIES.items():
    print(f"  [{k}]  {v}")

11 categories defined:
  [praise_admiration]  Positive, complimentary, or admiring reaction expressed in words (e.g. 'beautiful', 'amazing work', 'I love this', 'stunning').
  [emoji_only]  Comment consists entirely or almost entirely of emoji/symbols with no meaningful text (e.g. '😍❤️🔥', '👏👏👏').
  [authenticity_policing]  Questions or denies the legitimacy of the work as real art (e.g. 'this isn't real art', 'AI can't be creative', 'just pressing buttons').
  [ethics_copyright]  Raises concerns about ethics, copyright, or harm to human artists (e.g. 'you stole from artists', 'AI training is theft').
  [hostility_harassment]  Aggressive, rude, dismissive, or harassing tone directed at the creator or the work.
  [support_defense]  Defends AI art or the creator, or pushes back on criticism (e.g. 'AI is just a tool', 'all art borrows from others').
  [neutral_question]  Neutral comment or factual question with no strong sentiment (e.g. 'What app did you use?', 'What is Niji?', asking for 

## 5. LLM classification via Ollama (local)

In [6]:
def ollama_request(prompt: str, retries: int = 3) -> str:
    """Send a prompt to the local Ollama server. Returns response text."""
    payload = json.dumps({
        "model":  MODEL,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": 0.1, "num_predict": 200},
    }).encode()
    headers = {"Content-Type": "application/json"}
    for attempt in range(retries):
        try:
            req = urllib.request.Request(OLLAMA_URL, data=payload, headers=headers)
            with urllib.request.urlopen(req, timeout=120) as resp:
                return json.loads(resp.read())["response"].strip()
        except urllib.error.URLError:
            raise RuntimeError(
                "Cannot reach Ollama at localhost:11434.\n"
                "  -> Is Ollama installed? Download from https://ollama.com\n"
                f" -> Is the model pulled? Run: ollama pull {MODEL}\n"
                "  -> Is the server running? Run: ollama serve"
            )
        except Exception as e:
            wait = 2 ** attempt
            print(f"  Retrying in {wait}s ({e})")
            time.sleep(wait)
    raise RuntimeError("Max retries exceeded")


# Debug: confirm Ollama is running
print(f"Testing '{MODEL}' at {OLLAMA_URL} ...")
try:
    print(f"SUCCESS — {ollama_request('Reply with the single word: hello')}")
except Exception as e:
    print(f"FAILED:\n{e}")

Testing 'llama3.2' at http://localhost:11434/api/generate ...
SUCCESS — hello


In [7]:
CAT_KEYS_STR = ", ".join(f'"{k}"' for k in CATEGORY_KEYS)

# Regex: comment is just @tags and/or whitespace — no actual content
TAG_ONLY_RE = re.compile(r'^(\s*@\w+\s*)+$')

def build_prompt(comment: str) -> str:
    return (
        "You are a content analyst classifying social-media comments about artwork.\n"
        "Output JSON only — no markdown, no extra text.\n\n"
        "CATEGORIES (pick exactly one):\n"
        "  emoji_only           — comment contains ONLY emoji/symbols, zero letters or words\n"
        "  praise_admiration    — expresses genuine admiration or positivity IN WORDS\n"
        "                         (includes excited language, mild profanity, compliments,\n"
        "                          or emoji mixed with words like 'So beautiful ❤️')\n"
        "  authenticity_policing — disputes whether this counts as real/legitimate art\n"
        "                         ONLY use if the comment explicitly questions the art's legitimacy\n"
        "  ethics_copyright     — raises concern about ethics, IP, or harm to human artists\n"
        "  hostility_harassment  — aggressive, rude, or harassing toward creator or others\n"
        "  support_defense      — defends AI art or pushes back on criticism of it\n"
        "  neutral_question     — neutral observation or genuine question, no strong sentiment\n"
        "  humor_meme           — ONLY use for clear jokes, puns, or named meme references\n"
        "                         NOT for enthusiasm, excitement, or strong praise\n"
        "  needs_review         — comment meaning is genuinely unclear (e.g. ambiguous slang,\n"
        "                         mixed languages, inside joke with no clear sentiment)\n"
        "  non_english          — comment is clearly written in a non-English language\n"
        "  other                — meaning is clear but fits none of the above\n\n"
        "DECISION RULES — apply in order:\n"
        "  1. No letters/words at all? → emoji_only\n"
        "  2. Clearly written in another language (not English)? → non_english\n"
        "  3. Admires or praises the work, even with exclamations? → praise_admiration\n"
        "  4. Clear joke, pun, or meme with a punchline? → humor_meme\n"
        "  5. Genuinely cannot tell what the comment means? → needs_review\n"
        "  6. Otherwise follow the remaining category definitions above\n\n"
        "IMPORTANT:\n"
        "  - @username mentions are just social media tags — ignore them when reading the comment\n"
        "  - Do NOT use authenticity_policing just because a comment is confusing or foreign\n"
        "  - If the comment is in another language → non_english\n"
        "  - Only use needs_review if the language is English but meaning is still unclear\n\n"
        f"Comment: \"{comment}\"\n\n"
        '{"category": "<key>", "confidence": "high|medium|low", "reasoning": "<5 words>"}'
    )


import unicodedata

def is_emoji_only(text: str) -> bool:
    """Return True if the text contains no letters or digits — only emoji,
    punctuation, symbols, and whitespace."""
    for ch in text:
        cat = unicodedata.category(ch)
        if cat.startswith('L') or cat.startswith('N'):
            return False
    return len(text.strip()) > 0


def classify_comment(comment: str) -> dict:
    # 1. Pure emoji — no LLM call needed
    if is_emoji_only(comment):
        return {"category": "emoji_only", "confidence": "high", "reasoning": "no letters or digits"}
    # 2. Only @tags and whitespace — no content to classify
    if TAG_ONLY_RE.match(comment):
        return {"category": "other", "confidence": "high", "reasoning": "tag only, no content"}
    try:
        raw = ollama_request(build_prompt(comment))
        match = re.search(r'\{[^{}]+\}', raw, re.DOTALL)
        if not match:
            return {"category": "needs_review", "confidence": "low", "reasoning": "no JSON in response"}
        result = json.loads(match.group())
        if result.get("category") not in CATEGORY_KEYS:
            result["category"] = "needs_review"
        return result
    except (json.JSONDecodeError, KeyError):
        return {"category": "needs_review", "confidence": "low", "reasoning": "parse error"}
    except Exception as e:
        print(f"  Error: {e}")
        return {"category": "needs_review", "confidence": "low", "reasoning": str(e)[:100]}


# Smoke test
print("Smoke test:")
tests = [
    ("This isn't real art, you just pressed a button.", "authenticity_policing"),
    ("😍😍😍❤️❤️🔥",                                    "emoji_only"),
    ("🔥❤️",                                            "emoji_only"),
    ("So beautiful ❤️",                                 "praise_admiration"),
    ("Absofuckenlutely love it!",                       "praise_admiration"),
    ("What tool did you use for this?",                 "neutral_question"),
    ("@username @otherfriend",                          "other"),
    ("@caffe1ner uite dinasta merge sa ne luam",        "non_english"),
    ("Dracuszardu",                                     "needs_review"),
]
all_pass = True
for comment, expected in tests:
    result = classify_comment(comment)
    got  = result['category']
    mark = '✓' if got == expected else '✗'
    if got != expected:
        all_pass = False
    print(f"  {mark} [{expected:25s}] got [{got}]  — '{comment[:50]}'")
print()
print('All tests passed ✓' if all_pass else 'Some tests failed — consider tweaking the prompt.')

Smoke test:
  ✓ [authenticity_policing    ] got [authenticity_policing]  — 'This isn't real art, you just pressed a button.'
  ✓ [emoji_only               ] got [emoji_only]  — '😍😍😍❤️❤️🔥'
  ✓ [emoji_only               ] got [emoji_only]  — '🔥❤️'
  ✓ [praise_admiration        ] got [praise_admiration]  — 'So beautiful ❤️'
  ✓ [praise_admiration        ] got [praise_admiration]  — 'Absofuckenlutely love it!'
  ✓ [neutral_question         ] got [neutral_question]  — 'What tool did you use for this?'
  ✓ [other                    ] got [other]  — '@username @otherfriend'
  ✓ [non_english              ] got [non_english]  — '@caffe1ner uite dinasta merge sa ne luam'
  ✓ [needs_review             ] got [needs_review]  — 'Dracuszardu'

All tests passed ✓


In [ ]:
classify_df = all_df.copy()
results = []

for _, row in tqdm(classify_df.iterrows(), total=len(classify_df), desc="Classifying"):
    results.append(classify_comment(row["comment"]))
    time.sleep(SLEEP_BETWEEN_CALLS)

res_df = pd.DataFrame(results)
classify_df = pd.concat([classify_df.reset_index(drop=True), res_df], axis=1)

print(f"\nDone! {len(classify_df)} comments classified.")
classify_df[["condition", "post_id", "comment", "category", "confidence"]].head(12)

Classifying:   0%|          | 0/3317 [00:00<?, ?it/s]


Done! 3317 comments classified.


,condition,post_id,comment,category,confidence
0,AI,1,So beautiful ❣️,praise_admiration,high
1,AI,1,Simply outstanding 🔥🔥🔥,praise_admiration,high
2,AI,1,"Oh darling, say👏it👏louder👏. Your vibe and art ...",praise_admiration,high
3,AI,1,❤️❤️❤️,emoji_only,high
4,AI,1,Adore this job 😍 sea horse so beautiful 💫,praise_admiration,high
5,AI,1,That's why the kids have to be in bed! 😮👏👏,praise_admiration,high
6,AI,1,🤍🤍🌟🙌🖤,emoji_only,high
7,AI,1,😮,emoji_only,high
8,AI,1,😍😍😍,emoji_only,high
9,AI,1,🔥,emoji_only,high


## 6. Save classified data

In [ ]:
OUT_PATH = "data/classified_comments.csv"
classify_df.to_csv(OUT_PATH, index=False)
print(f"Saved to {OUT_PATH}")
print("\nCategory distribution overall:")
print(classify_df["category"].value_counts().to_string())

Saved to classified_comments.csv

Category distribution overall:
category
praise_admiration        1616
emoji_only               1293
non_english               147
humor_meme                 98
needs_review               71
other                      43
support_defense            16
hostility_harassment       15
neutral_question           10
authenticity_policing       8
